In [1]:
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict


# =========================================================
# 1. STATE
# =========================================================

class State(TypedDict):
    count: int


# =========================================================
# 2. NODES
# =========================================================

def add_one(state: State):

    print("ADD_ONE running...")

    return {
        "count": state["count"] + 1
    }


def add_two(state: State):

    print("ADD_TWO running...")

    return {
        "count": state["count"] + 2
    }


def add_three(state: State):

    print("ADD_THREE running...")

    return {
        "count": state["count"] + 3
    }


# =========================================================
# 3. BUILD GRAPH
# =========================================================

builder = StateGraph(State)

builder.add_node("add_one", add_one)
builder.add_node("add_two", add_two)
builder.add_node("add_three", add_three)

builder.add_edge(START, "add_one")
builder.add_edge("add_one", "add_two")
builder.add_edge("add_two", "add_three")
builder.add_edge("add_three", END)


# =========================================================
# 4. CHECKPOINTER
# =========================================================

checkpointer = InMemorySaver()

graph = builder.compile(
    checkpointer=checkpointer
)


# =========================================================
# 5. ORIGINAL THREAD
# =========================================================

original_config = {
    "configurable": {
        "thread_id": "thread-001"
    }
}


print("\n========================================")
print("ORIGINAL EXECUTION")
print("========================================")


original_result = graph.invoke(
    {"count": 0},
    config=original_config
)

print(
    "\nOriginal Result:",
    original_result
)


# =========================================================
# 6. GET ORIGINAL HISTORY
# =========================================================

history = list(
    graph.get_state_history(
        original_config
    )
)


print("\n========================================")
print("ORIGINAL CHECKPOINT HISTORY")
print("========================================")


for index, state in enumerate(history):

    print(
        f"\nCheckpoint {index}"
    )

    print(
        "Checkpoint ID:",
        state.config[
            "configurable"
        ]["checkpoint_id"]
    )

    print(
        "Values:",
        state.values
    )

    print(
        "Next:",
        state.next
    )


# =========================================================
# 7. SELECT HISTORICAL CHECKPOINT
# =========================================================

# History normally newest -> oldest.
#
# We need a checkpoint which has
# a state from which we want to create
# another execution path.

target_state = None

for state in history:

    if state.next:

        target_state = state
        break


if target_state is None:

    print(
        "\nNo suitable checkpoint found."
    )

else:

    old_checkpoint_id = (
        target_state.config[
            "configurable"
        ]["checkpoint_id"]
    )


    # =====================================================
    # 8. READ HISTORICAL CHECKPOINT
    # =====================================================

    old_checkpoint_config = {
        "configurable": {
            "thread_id": "thread-001",
            "checkpoint_id": old_checkpoint_id
        }
    }


    old_state = graph.get_state(
        old_checkpoint_config
    )


    print("\n========================================")
    print("HISTORICAL CHECKPOINT")
    print("========================================")

    print(
        "Thread ID:",
        old_state.config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Checkpoint ID:",
        old_checkpoint_id
    )

    print(
        "Historical State:",
        old_state.values
    )

    print(
        "Next Node:",
        old_state.next
    )


    # =====================================================
    # 9. CREATE FORKED THREAD
    # =====================================================

    fork_config = {
        "configurable": {
            "thread_id": "thread-002"
        }
    }


    # =====================================================
    # 10. USE HISTORICAL STATE
    #     AS FORK STARTING STATE
    # =====================================================

    fork_starting_state = {
        "count": old_state.values["count"]
    }


    print("\n========================================")
    print("FORKED THREAD")
    print("========================================")

    print(
        "Fork Thread ID:",
        fork_config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Fork Starting State:",
        fork_starting_state
    )


    # =====================================================
    # 11. ALTERNATE EXECUTION
    # =====================================================

    fork_result = graph.invoke(
        fork_starting_state,
        config=fork_config
    )


    print("\n========================================")
    print("FORK RESULT")
    print("========================================")

    print(
        "Fork Result:",
        fork_result
    )


    # =====================================================
    # 12. GET FORKED THREAD STATE
    # =====================================================

    fork_final_state = graph.get_state(
        fork_config
    )


    print("\n========================================")
    print("FORK FINAL STATE")
    print("========================================")

    print(
        "Thread ID:",
        fork_final_state.config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Values:",
        fork_final_state.values
    )

    print(
        "Checkpoint ID:",
        fork_final_state.config[
            "configurable"
        ]["checkpoint_id"]
    )


    # =====================================================
    # 13. ORIGINAL THREAD AGAIN
    # =====================================================

    original_final_state = graph.get_state(
        original_config
    )


    # =====================================================
    # 14. FINAL COMPARISON
    # =====================================================

    print("\n========================================")
    print("ORIGINAL VS FORK")
    print("========================================")

    print("\nORIGINAL THREAD")

    print(
        "Thread ID:",
        original_config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Final State:",
        original_final_state.values
    )

    print(
        "Final Checkpoint:",
        original_final_state.config[
            "configurable"
        ]["checkpoint_id"]
    )


    print("\nFORKED THREAD")

    print(
        "Thread ID:",
        fork_config[
            "configurable"
        ]["thread_id"]
    )

    print(
        "Final State:",
        fork_final_state.values
    )

    print(
        "Final Checkpoint:",
        fork_final_state.config[
            "configurable"
        ]["checkpoint_id"]
    )


ORIGINAL EXECUTION
ADD_ONE running...
ADD_TWO running...
ADD_THREE running...

Original Result: {'count': 6}

ORIGINAL CHECKPOINT HISTORY

Checkpoint 0
Checkpoint ID: 1f1bcffa-09d0-6d2f-8003-206bf4b7c9e8
Values: {'count': 6}
Next: ()

Checkpoint 1
Checkpoint ID: 1f1bcffa-09cc-6187-8002-b97f3c359a65
Values: {'count': 3}
Next: ('add_three',)

Checkpoint 2
Checkpoint ID: 1f1bcffa-09c7-6ff1-8001-cbde833ba552
Values: {'count': 1}
Next: ('add_two',)

Checkpoint 3
Checkpoint ID: 1f1bcffa-09c2-6d8e-8000-6fe974bc6c58
Values: {'count': 0}
Next: ('add_one',)

Checkpoint 4
Checkpoint ID: 1f1bcffa-09bc-6f73-bfff-a5e4fc8e9e83
Values: {}
Next: ('__start__',)

HISTORICAL CHECKPOINT
Thread ID: thread-001
Checkpoint ID: 1f1bcffa-09cc-6187-8002-b97f3c359a65
Historical State: {'count': 3}
Next Node: ('add_three',)

FORKED THREAD
Fork Thread ID: thread-002
Fork Starting State: {'count': 3}
ADD_ONE running...
ADD_TWO running...
ADD_THREE running...

FORK RESULT
Fork Result: {'count': 9}

FORK FINAL STATE
T